# Instance Generator for Robust Lot-Sizing

Generates synthetic Markov-modulated demand instances for testing robust optimization algorithms.

## Overview

This notebook creates test instances with configurable parameters across multiple dimensions:

- **Capacity**: Production capacity constraints
- **Costs**: Production, setup, holding, backlog penalties
- **Demand**: Normal vs. disaster states with correlation structure
- **Uncertainty**: Demand variability and transition probability bounds

## Instance Families

Generates instances for different horizons: T ∈ {12, 24, 36, 48}

Control parameters can be customized to explore:
- Cost structures (backlog vs. holding tradeoffs)
- Demand regimes (normal, disaster, hybrid)
- Transition dynamics (disaster persistence, state stickiness)
- Uncertainty bounds (conservatism parameters)

## Output

Saves instances to JSON files with all parameters embedded.
Each instance includes:
- T, L (horizon, # of states)
- C, p, h, b, q (capacity and costs)
- Mean/std of demand for each state
- P0 (transition probability matrix)
- Uncertainty parameters (Gamma_d, Gamma_P, alpha)

## Usage Example

```python
control_params = {
    'Capacity_Avg': 100,
    'Cost_p': 20,
    'DemandRatio_D': 2.0,
    'P0_NN': 0.8
}

generate_instance(T=24, control_params=control_params, instance_name='my_instance')
```

In [ ]:
import numpy as np
import random
import json
import os

In [ ]:
def generate_instance(T, control_params, instance_name="instance"):
    """Generates detailed parameters based on control parameters and saves to JSON."""
    L = 2 # Fixed
    
    # --- Generate Capacity ---
    cap_avg = control_params.get('Capacity_Avg', 100)
    cap_var = control_params.get('Capacity_Var', 0.1)
    C = random.uniform(cap_avg * (1 - cap_var), cap_avg * (1 + cap_var))
    C = int(round(C)) # Make capacity integer

    # --- Generate Costs ---
    p_base = control_params.get('Cost_p', 20)
    h_ratio = control_params.get('CostRatio_h_p', 0.1) # h = 10% of p
    b_ratio = control_params.get('CostRatio_b_h', 10) # b = 10 * h
    q_ratio = control_params.get('CostRatio_q_pc', 0.5) # q = 50% of p*C

    p = [p_base for _ in range(T)]
    h_val = max(1, round(p_base * h_ratio)) # Ensure h >= 1
    h = [h_val for _ in range(T)]
    b_val = max(1, round(h_val * b_ratio)) # Ensure b >= 1
    b = [b_val for _ in range(T)]
    q_val = max(1, round(p_base * C * q_ratio)) # Ensure q >= 1
    q = [q_val for _ in range(T)]

    # --- Generate Demand Parameters ---
    ratio_n = control_params.get('DemandRatio_N', 0.6)
    ratio_d = control_params.get('DemandRatio_D', 2.0) # Disaster demand higher than capacity
    cov_n = control_params.get('CoV_N', 0.2)
    cov_d = control_params.get('CoV_D', 0.3)

    mean_n = cap_avg * ratio_n
    mean_d = cap_avg * ratio_d
    # Ensure means and std devs are non-negative
    mean_n = max(0, mean_n); mean_d = max(0, mean_d)
    std_n = max(0, mean_n * cov_n)
    std_d = max(0, mean_d * cov_d)


    # Assign means/stds to the correct state index (0=Disaster, 1=Normal)
    mean_normal_demand_list = [[0.0, mean_n] for _ in range(T)]
    std_normal_demand_list  = [[0.0, std_n] for _ in range(T)]
    mean_disaster_demand_list = [[mean_d, 0.0] for _ in range(T)]
    std_disaster_demand_list  = [[std_d, 0.0] for _ in range(T)]

    # --- Generate Transition Parameters ---
    p0_dd = control_params.get('P0_DD', 0.4)
    p0_nn = control_params.get('P0_NN', 0.8)
    # Ensure probabilities are valid
    p0_dd = max(0.0, min(1.0, p0_dd))
    p0_nn = max(0.0, min(1.0, p0_nn))
    P0_list = [[p0_dd, 1.0 - p0_dd],
                [1.0 - p0_nn, p0_nn]]

    sigma_p_d = control_params.get('SigmaP_D', 0.1)
    sigma_p_n = control_params.get('SigmaP_N', 0.05)
    sigma_double_prime_list = [sigma_p_d, sigma_p_n]

    # --- Other Parameters ---
    Gamma_d = control_params.get('Gamma_d', 1.0)
    Gamma_P = control_params.get('Gamma_P', 1.0)
    alpha   = control_params.get('alpha', 0.01)
    epsilon = control_params.get('epsilon', 0.05)

    # --- Collate Parameters ---
    params = {
        "T": T,
        "L": L,
        "mean_normal_demand": mean_normal_demand_list,
        "std_normal_demand": std_normal_demand_list,
        "mean_disaster_demand": mean_disaster_demand_list,
        "std_disaster_demand": std_disaster_demand_list,
        "P0": P0_list,
        "sigma_double_prime": sigma_double_prime_list,
        "Gamma_d": Gamma_d,
        "Gamma_P": Gamma_P,
        "alpha": alpha,
        "C": C,
        "p": p,
        "b": b,
        "h": h,
        "q": q,
        "epsilon": epsilon
    }

    # --- Save to File ---
    output_dir = "instances"
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.join(output_dir, f"{instance_name}.json")
    try:
        with open(filename, 'w') as f:
            json.dump(params, f, indent=4)
        print(f"Instance saved successfully to {filename}")
    except Exception as e:
        print(f"Error saving instance to {filename}: {e}")

    return params 

In [ ]:
# --- Example Usage ---
if __name__ == "__main__":

    # Define the BASELINE control parameters (Medium settings)
    control_base = {
        'Capacity_Avg': 90, 'DemandRatio_N': 80/90, 'DemandRatio_D': 300/90,
        'CoV_N': 20/80, 'CoV_D': 60/300, 'Cost_p': 22, 'CostRatio_h_p': 7/22,
        'CostRatio_b_h': 70/7, 'CostRatio_q_pc': 0.15, 'P0_DD': 0.5,
        'P0_NN': 0.7, # Corresponds to P0_ND = 0.3
        'SigmaP_D': 0.1, 'SigmaP_N': 0.05, 'Gamma_d': 1.0,
        'Gamma_P': 1.0, 'alpha': 0.01, 'epsilon': 0.05
    }
    # Note: P0_ND is derived from P0_NN inside the function, or modify generator
    # to accept P0_ND directly if preferred.

    # --- Define Sensitivity Analysis Sets ---
    # Start with base, change one parameter at a time

    # Backlog Cost Ratio Variations
    control_bh_low  = control_base.copy(); control_bh_low['CostRatio_b_h'] = 5
    control_bh_high = control_base.copy(); control_bh_high['CostRatio_b_h'] = 20

    # Demand Budget Variations
    control_gd_low  = control_base.copy(); control_gd_low['Gamma_d'] = 0.5
    control_gd_high = control_base.copy(); control_gd_high['Gamma_d'] = 2.0

    # Transition Budget Variations
    control_gp_low  = control_base.copy(); control_gp_low['Gamma_P'] = 0.2
    control_gp_high = control_base.copy(); control_gp_high['Gamma_P'] = 1.8

    # N->D Probability Variations (adjusting P0_NN)
    # High Correlation: Both states are "sticky". Disasters last long.
    control_corr_high = control_base.copy()
    control_corr_high['P0_NN'] = 0.9  # Hard to enter disaster
    control_corr_high['P0_DD'] = 0.8  # Hard to leave disaster (Sticky!)

    # Low Correlation: System flips randomly (high entropy)
    control_corr_low = control_base.copy()
    control_corr_low['P0_NN'] = 0.5
    control_corr_low['P0_DD'] = 0.5   # Coin flip persistence

    # Alpha Variations
    control_alpha_low  = control_base.copy(); control_alpha_low['alpha'] = 0.001
    control_alpha_high = control_base.copy(); control_alpha_high['alpha'] = 0.05

    # Disaster Demand Variability Variations
    control_covd_low  = control_base.copy(); control_covd_low['CoV_D'] = 0.1
    control_covd_high = control_base.copy(); control_covd_high['CoV_D'] = 0.4
    
    control_all_low = control_base.copy() # Start from base
    control_all_low['CostRatio_b_h'] = 5     # Low backlog penalty
    control_all_low['Gamma_d'] = 0.5         # Low demand budget
    control_all_low['Gamma_P'] = 0.2         # Low transition budget
    control_all_low['P0_NN'] = 0.9           # Corresponds to LOW N->D probability (0.1)
    control_all_low['P0_DD'] = 0.3           # Low disaster persistence
    control_all_low['alpha'] = 0.001         # LOW threshold (allows less likely paths)
    control_all_low['CoV_N'] = 0.1           # Low normal variability (Added this for completeness)
    control_all_low['CoV_D'] = 0.1           # Low disaster variability

    control_all_high = control_base.copy() # Start from base
    control_all_high['CostRatio_b_h'] = 20    # High backlog penalty
    control_all_high['Gamma_d'] = 2.0         # High demand budget
    control_all_high['Gamma_P'] = 1.8         # High transition budget
    control_all_high['P0_NN'] = 0.5           # Corresponds to HIGH N->D probability (0.5)
    control_all_high['P0_DD'] = 0.7           # High disaster persistence
    control_all_high['alpha'] = 0.05          # HIGH threshold (restricts to more likely paths)
    control_all_high['CoV_N'] = 0.4           # High normal variability (Added this for completeness)
    control_all_high['CoV_D'] = 0.4           # High disaster variability

    # --- List or Dictionary of all instances to generate ---
    instances_to_generate = {
        "Base": control_base,
        "BH_Low": control_bh_low,
        "BH_High": control_bh_high,
        "Gd_Low": control_gd_low,
        "Gd_High": control_gd_high,
        "Gp_Low": control_gp_low,
        "Gp_High": control_gp_high,
        "Corr_High": control_corr_high, # Replaces P0ND_Low
        "Corr_Low": control_corr_low,   # Replaces P0ND_High
        "Alpha_Low": control_alpha_low,
        "Alpha_High": control_alpha_high,
        "CoV_D_Low": control_covd_low,
        "CoV_D_High": control_covd_high,
        "AllLow": control_all_low,
        "AllHigh": control_all_high
    }

    # --- Loop to generate instances for different horizons ---
    horizons = [12, 24, 36, 48]  # Add 24 (or 36, 48) here
    
    for T_horizon in horizons:
        print(f"\n{'='*40}")
        print(f"Generating instances for T={T_horizon}...")
        print(f"{'='*40}")
    
        for name, controls in instances_to_generate.items():
            instance_filename = f"{name}_T{T_horizon}"
            print(f"Generating {instance_filename}...")
            generate_instance(T=T_horizon, control_params=controls, instance_name=instance_filename)

    print("\nAll instances generated successfully.")


Generating instances for T=12...
Generating Base_T12...
Instance saved successfully to instances\Base_T12.json
Generating BH_Low_T12...
Instance saved successfully to instances\BH_Low_T12.json
Generating BH_High_T12...
Instance saved successfully to instances\BH_High_T12.json
Generating Gd_Low_T12...
Instance saved successfully to instances\Gd_Low_T12.json
Generating Gd_High_T12...
Instance saved successfully to instances\Gd_High_T12.json
Generating Gp_Low_T12...
Instance saved successfully to instances\Gp_Low_T12.json
Generating Gp_High_T12...
Instance saved successfully to instances\Gp_High_T12.json
Generating Corr_High_T12...
Instance saved successfully to instances\Corr_High_T12.json
Generating Corr_Low_T12...
Instance saved successfully to instances\Corr_Low_T12.json
Generating Alpha_Low_T12...
Instance saved successfully to instances\Alpha_Low_T12.json
Generating Alpha_High_T12...
Instance saved successfully to instances\Alpha_High_T12.json
Generating CoV_D_Low_T12...
Instance s

## Notes

- Instances are saved to JSON format for easy loading
- Each instance is self-contained (all parameters included)
- Seed-based generation ensures reproducibility
- Suitable for testing Benders decomposition and robust optimization algorithms